<a href="https://colab.research.google.com/github/mayait/CursoAnalisisDatos_IA_2026/blob/main/sitio/labs/lab_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Laboratorio 4 · Anatomía y calidad de los datos

La semana pasada describiste el negocio y dejaste tres cosas anotadas «para después»: cantidades
negativas, un precio de 59,10 en un catálogo cuyo producto más caro cuesta 5,91, y columnas con
`count` distinto. Hoy es después. Vas a poner un porcentaje exacto a cada problema de `ventas.csv`,
decidir qué hacer con cada uno y escribir la bitácora que permite a otra persona auditar tus
decisiones. Al final demuestras con una cifra de control que no perdiste información por el camino.
Esto desbloquea la semana 5: no se pueden unir tablas sucias sin multiplicar la suciedad.

> **Hoy haces** · Inventario de calidad de la base con el porcentaje exacto de cada problema
> (90 min). Arreglas las fechas mixtas, los duplicados, los nulos, las cantidades negativas y los
> precios rotos, con una justificación escrita por decisión. Produces la bitácora de limpieza y la
> cifra de control. Cierras encontrando los fallos silenciosos de un fragmento generado por IA.
>
> **Entrega** · Este cuaderno ejecutado, la bitácora de limpieza completa, la cifra de control
> cuadrada y el daño del fragmento con error cuantificado en dólares.
> Nombre de archivo: `lab_04_apellido.ipynb`.

In [ ]:
# --- Setup del entorno ---
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# Los datos de Comercial Andina viven en sitio/datos/
REPO = "https://github.com/mayait/CursoAnalisisDatos_IA_2026.git"
COPIA = Path("/content/CursoAnalisisDatos_IA_2026")
CANDIDATOS = [Path("../datos"), Path("datos"), Path("sitio/datos"),
              COPIA / "sitio" / "datos"]
DATOS = next((p for p in CANDIDATOS if p.exists()), None)
if DATOS is None:
    # En Colab el cuaderno llega solo: se trae el repositorio una sola vez.
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(COPIA)], check=True)
    DATOS = COPIA / "sitio" / "datos"

print("Setup completo ✓")
print(f"pandas {pd.__version__} · datos en {DATOS.resolve()}")

## 1. Inventario: cuantificar en lugar de opinar

«Los datos están sucios» no es un diagnóstico. Un diagnóstico dice **qué problema, cuántas filas y
qué porcentaje**, porque de ahí sale la decisión: un problema que afecta al 0,2 % se trata distinto
que uno que afecta al 12 %. Y antes de tocar nada se guardan las cifras de partida, porque sin ellas
no habrá cifra de control al final.

In [ ]:
crudo = pd.read_csv(DATOS / "ventas.csv")
clientes = pd.read_csv(DATOS / "clientes.csv")
productos = pd.read_csv(DATOS / "productos.csv")

# Cifras de partida: se guardan ANTES de cualquier cambio, para poder cuadrar al final.
N_FILAS_0 = len(crudo)
N_FACTURAS_0 = crudo["factura_id"].nunique()
N_PRODUCTOS_0 = crudo["producto_id"].nunique()
N_SUCURSALES_0 = crudo["sucursal_id"].nunique()
crudo["monto_bruto"] = crudo["cantidad"] * crudo["precio_unitario"] * (1 - crudo["descuento"])
MONTO_0 = crudo["monto_bruto"].sum()

print(f"filas {N_FILAS_0:,} · facturas {N_FACTURAS_0:,} · productos {N_PRODUCTOS_0} "
      f"· sucursales {N_SUCURSALES_0} · monto {MONTO_0:,.2f}\n")

radiografia = pd.DataFrame({
    "tipo": crudo.dtypes.astype(str),
    "no nulos": crudo.notna().sum(),
    "nulos": crudo.isna().sum(),
    "% nulos": (crudo.isna().mean() * 100).round(2),
    "valores distintos": crudo.nunique(),
})
print(radiografia.to_string())
print()

fecha_texto = crudo["fecha"].str.contains("/")
precios = crudo.merge(productos[["producto_id", "precio_lista"]], on="producto_id", how="left")
precio_x10 = ((precios["precio_unitario"] / precios["precio_lista"]) > 5).fillna(False)
cantidad_mala = (crudo["cantidad"] < 0) & (~crudo["es_devolucion"])

inventario = pd.DataFrame([
    ("cliente_id ausente",                        int(crudo["cliente_id"].isna().sum())),
    ("fecha escrita en dd/mm/aaaa",               int(fecha_texto.sum())),
    ("filas duplicadas exactas",                  int(crudo.duplicated().sum())),
    ("devolución legítima (marcada)",             int(crudo["es_devolucion"].sum())),
    ("cantidad negativa SIN marca de devolución", int(cantidad_mala.sum())),
    ("precio_unitario ausente",                   int(crudo["precio_unitario"].isna().sum())),
    ("precio con un cero de más",                 int(precio_x10.sum())),
], columns=["problema", "filas afectadas"])
inventario["% de las filas"] = (inventario["filas afectadas"] / N_FILAS_0 * 100).round(3)

con_problema = (crudo["cliente_id"].isna() | fecha_texto | crudo.duplicated(keep=False)
                | cantidad_mala | crudo["precio_unitario"].isna() | precio_x10.values)
print(f"filas con al menos un problema: {con_problema.sum():,} ({con_problema.mean():.1%})\n")

orden = inventario.sort_values("filas afectadas", ascending=False, ignore_index=True)
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.barh(orden["problema"], orden["% de las filas"], color="#C44E52")
ax.invert_yaxis()
ax.set_title("Ningún problema pasa del 12 %, pero entre todos tocan el 21,8 % de las filas")
ax.set_xlabel("% de las filas del archivo")
for y, valor in enumerate(orden["% de las filas"]):
    ax.text(valor + 0.15, y, f"{valor:.2f} %", va="center", fontsize=9)
plt.tight_layout()
plt.show()

orden

📌 Ningún problema pasa del 12 % y, aun así, el 21,8 % de las filas tiene al menos uno. Esa es la
forma típica de un archivo real: no está roto, está astillado. Fíjate en el orden de magnitud, porque
decide el tratamiento: los 160 precios con un cero de más son el 0,199 % de las filas y ya verás
cuánto dinero mueven.

## 2. Tipos de dato: `dtypes` antes que nada

`dtypes` es el primer método que se mira en un archivo nuevo, no el último. Dos tipos mal asignados
bastan para arruinar un análisis entero, y ninguno de los dos lanza un error.

In [ ]:
print("¿la fecha es una fecha?   ", crudo["fecha"].dtype)
print("¿el cliente es un número? ", crudo["cliente_id"].dtype)
print()
print("cliente_id como texto es CORRECTO: 'C00001' no es una cantidad, es una etiqueta.")
print("Si alguien lo convierte a número pierde los ceros y ya no cruza con clientes.csv:")
print("  '00001' →", int("00001"))

Un identificador **nunca** es un número, aunque solo tenga dígitos: no se suma, no se promedia y sus
ceros a la izquierda son parte del valor. La fecha, en cambio, sí tiene que dejar de ser texto.

## 3. Las fechas mixtas: dos formatos en la misma columna

In [ ]:
print(f"formato aaaa-mm-dd : {(~fecha_texto).sum():>6,}  ({(~fecha_texto).mean():.2%})")
print(f"formato dd/mm/aaaa : {fecha_texto.sum():>6,}  ({fecha_texto.mean():.2%})")
print("ejemplos aaaa-mm-dd:", crudo.loc[~fecha_texto, "fecha"].head(3).tolist())
print("ejemplos dd/mm/aaaa:", crudo.loc[fecha_texto, "fecha"].head(3).tolist())
print()

# Intento ingenuo: pandas infiere UN formato con las primeras filas y descarta el resto.
ingenuo = pd.to_datetime(crudo["fecha"], errors="coerce")
print(f"con errors='coerce' se convierten en NaT : {ingenuo.isna().sum():,} ({ingenuo.isna().mean():.2%})")
print("y es exactamente el bloque dd/mm/aaaa    :", bool((ingenuo.isna() == fecha_texto).all()))

⚠️ Ese `errors="coerce"` es el fallo silencioso más caro de esta semana: **borra el 12 % del año sin
avisar**. Y hay una versión peor. Si el bloque `dd/mm/aaaa` se lee con el orden americano, pandas no
falla en las fechas imposibles —descarta `29/07/2024` porque no existe el mes 29— pero **sí acepta las
ambiguas**, y las manda al mes equivocado.

In [ ]:
dia = crudo.loc[fecha_texto, "fecha"].str[:2].astype(int)
ambiguas = dia <= 12
print(f"fechas dd/mm/aaaa con día ≤ 12 : {ambiguas.sum():,} de {fecha_texto.sum():,} ({ambiguas.mean():.1%})")
print(f"  '02/01/2026' como día/mes : {pd.to_datetime('02/01/2026', dayfirst=True):%d-%m-%Y}")
print(f"  '02/01/2026' como mes/día : {pd.to_datetime('02/01/2026', dayfirst=False):%d-%m-%Y}")
print("  Un mes de diferencia, sin error y sin aviso.\n")

# La conversión correcta: formato mixto, día primero, y verificación del rango resultante.
ventas = crudo.copy()
ventas["fecha"] = pd.to_datetime(ventas["fecha"], format="mixed", dayfirst=True)
N_DIAS_0 = ventas["fecha"].nunique()

print(f"tipo {ventas['fecha'].dtype} · fechas perdidas {ventas['fecha'].isna().sum()} "
      f"· rango {ventas['fecha'].min():%d-%m-%Y} a {ventas['fecha'].max():%d-%m-%Y} "
      f"· días con actividad {N_DIAS_0:,}")

Rango coherente y cero pérdidas. La verificación del rango es obligatoria: es lo único que distingue
«convertí las fechas» de «convertí las fechas bien».

## 4. Duplicados, y por qué el orden de las operaciones cambia el resultado

`duplicated()` marca filas idénticas en todas las columnas. Pero acabas de cambiar una columna, así
que el conteo ya no es el mismo que hace tres celdas.

In [ ]:
dup_texto = crudo.duplicated().sum()
dup_fecha = ventas.duplicated().sum()

print(f"duplicados con la fecha como texto      : {dup_texto:,}  ({dup_texto / N_FILAS_0:.2%})")
print(f"duplicados con la fecha ya convertida   : {dup_fecha:,}  ({dup_fecha / N_FILAS_0:.2%})")
print(f"filas que solo se delatan al normalizar : {dup_fecha - dup_texto:,}")

📌 Doscientas cuarenta filas eran duplicados de verdad y estaban escondidas detrás del formato de la
fecha: la misma línea de factura cargada dos veces, una como `2025-07-07` y otra como `07/07/2025`.
Si hubieras deduplicado antes de convertir, se te quedaban dentro. **Normalizar primero, deduplicar
después.**

`duplicated()` sin argumentos busca el duplicado exacto. Con `subset` buscas el duplicado *de
negocio*, que es otra cosa y casi nunca se elimina.

In [ ]:
dup_negocio = ventas.duplicated(subset=["factura_id", "producto_id"]).sum()

print(f"duplicado exacto (todas las columnas)     : {dup_fecha:,}")
print(f"duplicado de negocio (factura + producto) : {dup_negocio:,}")
print()
print("El segundo incluye al primero más los casos legítimos: el mismo producto en dos")
print("líneas de la misma factura, con cantidad o descuento distintos. Solo se eliminan los exactos.")

ventas = ventas.drop_duplicates().reset_index(drop=True)
print(f"\nfilas tras eliminar duplicados exactos: {len(ventas):,}")

## 5. Nulos: ausente no es cero, y desconocido tampoco

`cliente_id` falta en 6 273 líneas del archivo crudo. La tentación es rellenar con cero y seguir. La
pregunta correcta es **qué significa el hueco**: esas ventas ocurrieron, el dinero entró, lo único
que no sabemos es a quién atribuirlas. No son ventas de cero, ni ventas a un cliente llamado cero:
son ventas sin atribuir.

In [ ]:
sin_id = ventas["cliente_id"].isna()

print(f"líneas sin cliente_id : {sin_id.sum():,} ({sin_id.mean():.2%}) sobre la tabla ya deduplicada")
print(f"facturas afectadas    : {ventas.loc[sin_id, 'factura_id'].nunique():,}")
print(f"monto involucrado     : {ventas.loc[sin_id, 'monto_bruto'].sum():,.2f} "
      f"({ventas.loc[sin_id, 'monto_bruto'].sum() / MONTO_0:.2%} de la facturación)\n")

reparto = pd.DataFrame({
    "sin cliente %": ventas.loc[sin_id, "sucursal_id"].value_counts(normalize=True) * 100,
    "con cliente %": ventas.loc[~sin_id, "sucursal_id"].value_counts(normalize=True) * 100,
})
reparto["diferencia"] = reparto["sin cliente %"] - reparto["con cliente %"]
reparto.round(2)

La ausencia está repartida entre todas las sucursales, así que no es el problema de una tienda
concreta: es ruido de captura. **Decisión: se conservan las filas y se marca el hueco con un
centinela explícito.** Se conservan porque el dinero es real y cualquier total que las descarte queda
un 7,4 % corto. Se marcan con `SIN_IDENTIFICAR` en lugar de dejarlas en nulo, para que ningún
`groupby` posterior las esconda en silencio, y en lugar de cero, para que nadie las confunda con un
cliente.

In [ ]:
ventas["cliente_id"] = ventas["cliente_id"].fillna("SIN_IDENTIFICAR")

print(f"etiquetas distintas en cliente_id : {ventas['cliente_id'].nunique():,}")
print(f"  de las cuales clientes reales   : {ventas['cliente_id'].nunique() - 1:,}")
print(f"nulos restantes                   : {ventas['cliente_id'].isna().sum()}")

## 6. Cantidades negativas: devolución legítima frente a error de tecleo

En la tabla deduplicada quedan 2 333 líneas con cantidad negativa, y no todas son lo mismo. El dato
que las separa está en dos columnas que nadie mira juntas: la marca `es_devolucion` y el prefijo del
número de documento.

In [ ]:
signo = np.where(ventas["cantidad"] < 0, "cantidad negativa", "cantidad positiva")
documento = np.where(ventas["factura_id"].str.startswith("C"),
                     "C · nota de crédito", "F · factura de venta")

pd.crosstab([documento, ventas["es_devolucion"]], signo, margins=True)

La tabla es concluyente. Las 1 862 devoluciones son coherentes: nota de crédito, marca en `True` y
cantidad negativa, las tres cosas a la vez. Las otras 471 son **facturas de venta**, con la marca en
`False` y cantidad negativa: nadie emite una factura `F` para devolver mercadería. Es un signo mal
tecleado.

**Decisión: las devoluciones se conservan tal cual —el negocio necesita verlas— y las 471 negativas
sin marca se corrigen a valor absoluto, dejando rastro en una columna nueva.** Corregir y no eliminar,
porque la venta existió: lo que está mal es el signo, no el hecho.

In [ ]:
neg_erronea = (ventas["cantidad"] < 0) & (~ventas["es_devolucion"])

ventas["cantidad_limpia"] = np.where(neg_erronea, ventas["cantidad"].abs(), ventas["cantidad"])
ventas["corregida_cantidad"] = neg_erronea

print(f"filas corregidas                            : {neg_erronea.sum():,} ({neg_erronea.mean():.3%})")
print(f"devoluciones conservadas con signo negativo : {int(ventas['es_devolucion'].sum()):,}")
print(f"cantidades negativas sin marca que quedan   : "
      f"{int(((ventas['cantidad_limpia'] < 0) & (~ventas['es_devolucion'])).sum())}")

## 7. Precios: los ausentes y los que tienen un cero de más

Los dos problemas de precio se resuelven con la misma herramienta —el catálogo de productos— pero por
razones distintas. Primero comprueba que el catálogo sirve como referencia.

In [ ]:
ventas = ventas.merge(productos[["producto_id", "precio_lista"]], on="producto_id", how="left")
razon = ventas["precio_unitario"] / ventas["precio_lista"]

print("razón precio_unitario / precio_lista")
print(razon.round(1).value_counts(dropna=False).to_string())
print(f"\ncoincidencia exacta con el catálogo : {(razon.round(3) == 1).mean():.2%} de las filas")

El 99,39 % de las líneas cobra exactamente el precio de lista, y el resto se parte en dos grupos
limpios: las que valen diez veces el precio de lista y las que no tienen precio. No hay zona gris, y
por eso aquí el catálogo es una fuente de imputación defendible. En un negocio con precios negociados
esta decisión no valdría, y habría que decirlo.

In [ ]:
precio_inflado = (razon > 5).fillna(False)
precio_ausente = ventas["precio_unitario"].isna()

inflado_declarado = ventas.loc[precio_inflado, "monto_bruto"].sum()
inflado_correcto = (ventas.loc[precio_inflado, "cantidad"] * ventas.loc[precio_inflado, "precio_lista"]
                    * (1 - ventas.loc[precio_inflado, "descuento"])).sum()

print(f"precios con un cero de más : {precio_inflado.sum():,} filas ({precio_inflado.mean():.3%})")
print(f"  facturación que inflan   : {inflado_declarado - inflado_correcto:,.2f} "
      f"({(inflado_declarado - inflado_correcto) / MONTO_0:.2%} del total declarado)")
print(f"  precio máximo observado  : {ventas['precio_unitario'].max():,.2f} "
      f"contra {productos['precio_lista'].max():,.2f} de máximo en el catálogo")
print(f"\nprecios ausentes           : {precio_ausente.sum():,} filas ({precio_ausente.mean():.3%})")

**Decisión para los inflados: se reemplazan por el precio de lista.** Son 160 filas, el 0,202 % de la
tabla, y sin embargo inflan la facturación en 41 740,02, un 1,47 % del total. Es el caso de libro de
un problema minúsculo en filas y grande en dinero; por eso el inventario se mira en filas **y** en
impacto.

**Decisión para los ausentes: se imputan desde el catálogo y se marca la fila.** La alternativa
—eliminarlas— borraría 322 ventas que sí ocurrieron. La alternativa peor —rellenar con cero— las
convertiría en regalos.

In [ ]:
ventas["precio_limpio"] = np.where(precio_inflado, ventas["precio_lista"], ventas["precio_unitario"])
ventas["imputado_precio"] = ventas["precio_limpio"].isna()
ventas["precio_limpio"] = ventas["precio_limpio"].fillna(ventas["precio_lista"])
ventas["corregido_precio"] = precio_inflado

ventas["monto"] = ventas["cantidad_limpia"] * ventas["precio_limpio"] * (1 - ventas["descuento"])

print(f"precios reemplazados por el de lista : {int(ventas['corregido_precio'].sum()):,}")
print(f"precios imputados desde el catálogo  : {int(ventas['imputado_precio'].sum()):,}")
print(f"precios nulos restantes              : {ventas['precio_limpio'].isna().sum()}")
print(f"precio máximo ahora                  : {ventas['precio_limpio'].max():,.2f}")

## 8. Las ciudades de `clientes.csv`

Mismo principio, otra tabla. Aquí el problema no son nulos sino **la misma ciudad escrita de varias
formas**, que es lo que rompe cualquier `groupby` por ciudad.

In [ ]:
print(f"valores distintos en la columna ciudad : {clientes['ciudad'].nunique()}")
print(clientes["ciudad"].value_counts().head(10).to_string())

clientes["ciudad_original"] = clientes["ciudad"]
clientes["ciudad"] = (clientes["ciudad"]
                      .str.strip()                                # espacios al principio y al final
                      .str.title()                                # QUITO y quito → Quito
                      .replace({"Guayaquíl": "Guayaquil"}))       # el error de tildes, a mano
corregidas = clientes["ciudad"] != clientes["ciudad_original"]

print(f"\nvalores distintos después : {clientes['ciudad'].nunique()} → {sorted(clientes['ciudad'].unique())}")
print(f"registros corregidos      : {corregidas.sum()} ({corregidas.mean():.2%})")
print(f"clientes sin fecha_alta   : {clientes['fecha_alta'].isna().sum()} "
      f"({clientes['fecha_alta'].isna().mean():.2%})\n")

canonicas = sorted(clientes["ciudad"].unique())
pd.DataFrame({
    "antes de normalizar": clientes["ciudad_original"].value_counts().reindex(canonicas),
    "después de normalizar": clientes["ciudad"].value_counts().reindex(canonicas),
}).assign(recuperados=lambda d: d["después de normalizar"] - d["antes de normalizar"])

Quito pasa de 617 clientes a 674: cincuenta y siete clientes de Quito estaban fuera del grupo «Quito»
por un espacio o una mayúscula, y un informe por ciudad los habría reportado como categorías sueltas
de una fila cada una. `.str.strip().str.title()` resuelve casi todo; el `replace` explícito es para lo
que ninguna regla general arregla, como una tilde de más.

Queda un problema que **no** se arregla: el 3,00 % de los clientes no tiene fecha de alta. No se
inventa. Se documenta y se convierte en una limitación conocida de cualquier análisis de antigüedad.

## 9. La bitácora de limpieza

Una limpieza sin bitácora no es reproducible: es una opinión. Una fila por problema, siempre las
mismas columnas, incluida la de justificación, que es la que casi nadie escribe. Las cifras de cada
fila se miden **en el momento en que se aplica la decisión**, después de los pasos anteriores.

In [ ]:
bitacora = pd.DataFrame([
    dict(problema="Fecha en dos formatos", tabla="ventas", filas=int(fecha_texto.sum()),
         decision="Convertir con format='mixed', dayfirst=True",
         justificacion="El parseo ingenuo borra el 12 % del año o lo manda al mes equivocado"),
    dict(problema="Filas duplicadas exactas", tabla="ventas", filas=int(dup_fecha),
         decision="Eliminar, después de normalizar la fecha",
         justificacion="240 de ellas solo se detectan con la fecha ya convertida"),
    dict(problema="cliente_id ausente", tabla="ventas", filas=int(sin_id.sum()),
         decision="Conservar la fila y marcar con el centinela SIN_IDENTIFICAR",
         justificacion="La venta ocurrió; lo desconocido es la atribución, no el monto"),
    dict(problema="Cantidad negativa sin marca", tabla="ventas", filas=int(neg_erronea.sum()),
         decision="Convertir a valor absoluto y marcar corregida_cantidad",
         justificacion="El documento es factura F, no nota de crédito C: el signo está mal tecleado"),
    dict(problema="Precio con un cero de más", tabla="ventas", filas=int(precio_inflado.sum()),
         decision="Reemplazar por precio_lista del catálogo",
         justificacion="El 99,39 % de las líneas cobra exactamente el precio de lista"),
    dict(problema="precio_unitario ausente", tabla="ventas", filas=int(precio_ausente.sum()),
         decision="Imputar desde precio_lista y marcar imputado_precio",
         justificacion="Eliminar borraría 322 ventas reales; rellenar con cero las volvería regalos"),
    dict(problema="Ciudad escrita de varias formas", tabla="clientes", filas=int(corregidas.sum()),
         decision="strip + title + replace de 'Guayaquíl'",
         justificacion="25 variantes de 5 ciudades rompen cualquier groupby por ciudad"),
    dict(problema="fecha_alta ausente", tabla="clientes", filas=int(clientes["fecha_alta"].isna().sum()),
         decision="No tratar: se deja en nulo",
         justificacion="No hay fuente para imputarla; pasa a limitación conocida de la ficha"),
])
bitacora["% de su tabla"] = np.where(bitacora["tabla"] == "ventas",
                                     bitacora["filas"] / N_FILAS_0 * 100,
                                     bitacora["filas"] / len(clientes) * 100).round(2)

# El impacto en dinero de cada decisión, medido y no estimado.
paso_0 = MONTO_0
paso_1 = ventas["monto_bruto"].sum()                                                     # sin duplicados
paso_2 = (ventas["cantidad"] * ventas["precio_limpio"] * (1 - ventas["descuento"])).sum()  # precios
paso_3 = ventas["monto"].sum()                                                           # cantidades

reconciliacion = pd.DataFrame([
    ("facturación declarada en el archivo crudo", paso_0, np.nan),
    ("eliminar duplicados exactos", paso_1, paso_1 - paso_0),
    ("corregir precios (inflados e imputados)", paso_2, paso_2 - paso_1),
    ("corregir cantidades negativas mal tecleadas", paso_3, paso_3 - paso_2),
    ("TOTAL corregido", paso_3, paso_3 - paso_0),
], columns=["paso", "facturación acumulada", "efecto del paso"])

print(bitacora[["problema", "tabla", "filas", "% de su tabla", "decision"]].to_string(index=False))
print()
reconciliacion

## 10. La cifra de control

La pregunta que cierra cualquier limpieza es: **¿perdí información?** No se contesta con una
sensación. Se contesta con invariantes: cosas que tenían que quedar exactamente igual, y cosas cuyo
cambio tiene que estar explicado hasta el último centavo.

In [ ]:
control = pd.DataFrame([
    ("filas",                N_FILAS_0,      len(ventas)),
    ("facturas distintas",   N_FACTURAS_0,   ventas["factura_id"].nunique()),
    ("productos distintos",  N_PRODUCTOS_0,  ventas["producto_id"].nunique()),
    ("sucursales distintas", N_SUCURSALES_0, ventas["sucursal_id"].nunique()),
    ("días con actividad",   N_DIAS_0,       ventas["fecha"].nunique()),
], columns=["invariante", "antes", "después"])
control["cambio"] = control["después"] - control["antes"]
print(control.to_string(index=False))

columnas_finales = ["factura_id", "fecha", "cliente_id", "sucursal_id",
                    "producto_id", "cantidad_limpia", "precio_limpio", "descuento"]
cuadra = abs(reconciliacion["efecto del paso"].iloc[1:4].sum() - (paso_3 - paso_0)) < 0.01
print(f"\nnulos restantes en las columnas de trabajo : {int(ventas[columnas_finales].isna().sum().sum())}")
print(f"diferencia de facturación explicada        : {paso_3 - paso_0:,.2f} ({(paso_3 - paso_0) / paso_0:+.2%})")
print(f"¿la reconciliación cuadra al centavo?      : {cuadra}")

ticket_limpio = ventas[~ventas["es_devolucion"]].groupby("factura_id")["monto"].sum()
print("\nEl ticket de la semana 3, recalculado sobre la tabla limpia:")
print(f"  media   {ticket_limpio.mean():>10,.2f}   (era 180,84 sobre la tabla sucia)")
print(f"  mediana {ticket_limpio.median():>10,.2f}   (era 30,08)")
print(f"  mínimo  {ticket_limpio.min():>10,.2f}   (era -693,68)")
print(f"  máximo  {ticket_limpio.max():>10,.2f}   (era 2 887,59)")

Las cuatro invariantes que importan se mantienen: **17 675 facturas antes y después, 74 productos,
6 sucursales y 923 días con actividad**. Se perdieron 1 151 filas —el 1,43 %— y todas eran copias
exactas de otra fila que sigue en la tabla: ninguna factura, ningún producto y ningún día de
actividad desapareció. La facturación bajó 35 888,98 (un 1,26 %) y cada centavo de esa diferencia
está en la tabla de reconciliación.

Ese es el criterio: no «quedó limpio», sino **«todo lo que cambió está explicado y todo lo que no
tenía que cambiar sigue igual»**.

Y el ticket de la semana pasada sobrevive: media 180,42, mediana 30,13. La bimodalidad no era un
artefacto de la suciedad. Lo que desaparece es el ruido —el mínimo negativo y el máximo imposible—,
que es exactamente lo que se espera de una limpieza honesta: **quita el ruido y deja la señal donde
estaba.**

### 🌶️ Ejercicio 1 — Guiado

Repite el inventario de la sección 1 sobre `clientes.csv` completo: una tabla con problema, filas
afectadas y porcentaje exacto para todas las columnas del archivo. Después añade a la bitácora una
fila por cada problema nuevo que encuentres, con su decisión y su justificación.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: empieza por clientes.isna().sum() y clientes.nunique()
# Pista 2: mira también canal_captacion y tipo_cliente con value_counts(dropna=False)
# Pista 3: ¿hay cliente_id duplicados? Sería mucho más grave que las ciudades: ¿por qué?

### 🔥 Desafío — el fragmento generado por IA

Este es el código que devolvió un asistente al prompt *«escribe una función que limpie mi DataFrame de
ventas en pandas»*. Corre sin errores, es legible y tiene buen aspecto. Contiene **tres fallos
silenciosos**. Ejecútalo, encuéntralos y cuantifica el daño de cada uno en dólares y en filas.

In [ ]:
def limpiar_ventas(df):
    """Limpieza básica de un DataFrame de ventas."""
    df = df.copy()
    df = df.fillna(0)                                            # rellenar valores faltantes
    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")   # normalizar fechas
    df = df.drop_duplicates()                                    # quitar duplicados
    df["monto"] = df["cantidad"] * df["precio_unitario"] * (1 - df["descuento"])
    return df


ventas_ia = limpiar_ventas(pd.read_csv(DATOS / "ventas.csv"))
print(f"{len(ventas_ia):,} filas, sin nulos, sin duplicados. Impecable.")
ventas_ia.head(4)

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: ventas_ia.groupby("cliente_id")["monto"].sum().sort_values(ascending=False).head()
#          ¿quién es el cliente más grande de Comercial Andina según este código?
# Pista 2: ventas_ia["fecha"].isna().sum()  ¿cuántas ventas del negocio desaparecieron del calendario?
# Pista 3: compara ventas_ia[ventas_ia["precio_unitario"] == 0] contra el catálogo de productos.
# Entrega: tres cifras, una por fallo, cada una en dólares o en filas.

### 🎯 Reto en clase (15 min)

En equipos. Cada equipo escribe **una** línea de limpieza plausible pero equivocada sobre
`ventas.csv` —del estilo de las tres del fragmento anterior— y se la pasa al equipo de al lado. El
equipo receptor tiene diez minutos para encontrar el daño y ponerle una cifra. Gana el equipo cuya
línea tarde más en ser cuantificada, no el que la esconda mejor: si nadie logra medir el daño, la
línea no valía nada como trampa.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: la trampa buena no lanza excepciones. Candidatas: dropna() sin subset,
# astype(int) sobre una columna con nulos, un groupby que descarta nulos en silencio,
# o un filtro por fecha aplicado antes de convertir la columna a datetime.

## La trampa de hoy

⚠️ **Rellenar los nulos con cero por defecto.** Es la línea más frecuente de todo el código de
limpieza generado por IA y la que más caro sale, porque no falla: produce una tabla completa,
ordenada y falsa. En `ventas.csv` el mismo `fillna(0)` hace dos daños distintos según la columna, y
el `errors="coerce"` añade un tercero.

In [ ]:
ranking_ia = ventas_ia.groupby("cliente_id")["monto"].sum().sort_values(ascending=False)
cliente_falso, mayor_real = ranking_ia.loc[0], ranking_ia.iloc[1]
facturas_falsas = ventas_ia.loc[ventas_ia["cliente_id"] == 0, "factura_id"].nunique()

faltantes = crudo["precio_unitario"].isna()
catalogo = crudo.merge(productos[["producto_id", "precio_lista"]], on="producto_id", how="left")
perdido = (catalogo.loc[faltantes, "cantidad"] * catalogo.loc[faltantes, "precio_lista"]
           * (1 - catalogo.loc[faltantes, "descuento"])).sum()

dic = (ventas["fecha"].dt.year == 2025) & (ventas["fecha"].dt.month == 12)
dic_ia = (ventas_ia["fecha"].dt.year == 2025) & (ventas_ia["fecha"].dt.month == 12)
dic_ok, dic_mal = ventas.loc[dic, "monto"].sum(), ventas_ia.loc[dic_ia, "monto"].sum()

print("RANKING DE CLIENTES SEGÚN EL CÓDIGO CON fillna(0)")
print(ranking_ia.head(4).to_string())
print(f"\nEl 'cliente 0' no existe: son {facturas_falsas:,} facturas de clientes desconocidos "
      f"apiladas bajo una sola etiqueta.")
print(f"  factura {cliente_falso / mayor_real:.2f} veces más que el mayor cliente real")
print(f"  concentra el {cliente_falso / ranking_ia.sum():.2%} de la facturación\n")

daño = pd.DataFrame([
    ("fillna(0) sobre cliente_id", f"{facturas_falsas:,} facturas",
     f"inventa el cliente n.º 1 de la empresa, con {cliente_falso:,.2f}"),
    ("fillna(0) sobre precio_unitario", f"{int(faltantes.sum()):,} líneas",
     f"borra {perdido:,.2f} de facturación real"),
    ("errors='coerce' sobre la fecha", f"{int(ventas_ia['fecha'].isna().sum()):,} filas",
     f"diciembre-2025 cae de {dic_ok:,.2f} a {dic_mal:,.2f} ({dic_mal / dic_ok - 1:+.1%})"),
], columns=["línea del fragmento", "filas tocadas", "daño"])
print(daño.to_string(index=False))

Un cliente sin identificador **no compró cero: no sabemos quién es**. Un precio ausente **no es un
precio de cero: es un precio que no se registró**. Y una fecha que no se pudo convertir **no es una
venta que no ocurrió**. Cero es un valor; ausente es la falta de un valor. Confundirlos produce un
informe impecable y equivocado, que es exactamente el informe con el que abrió este curso.

Fíjate en la asimetría: el `fillna(0)` del `cliente_id` no cambia ni un centavo de la facturación
total —por eso ningún control de suma lo detecta— y a la vez inventa el mayor cliente de la empresa.
Los errores que no mueven el total son los que llegan más lejos.

## Entregable

Sube `lab_04_apellido.ipynb` con:

- El inventario de calidad completo, con el porcentaje exacto de cada problema.
- La bitácora de limpieza con las ocho filas, más las que añadas en el ejercicio guiado. Cada fila con
  su decisión **y** su justificación en una línea.
- La cifra de control: la tabla de invariantes y la reconciliación de la facturación cuadrada al
  centavo.
- Los tres fallos del fragmento generado por IA, cada uno con su daño cuantificado.
- Una fila nueva en la bitácora de prompts: el prompt con el que le pediste al asistente que revisara
  tu propia función de limpieza, y qué encontró que tú no habías visto.

## Para tu equipo

- La bitácora de limpieza del dataset del caso es entregable de esta semana. La columna de
  justificación es la que se califica: cualquiera elimina duplicados, casi nadie escribe por qué.
- Antes de limpiar, guarden las cifras de partida —filas, claves distintas, suma de la columna de
  dinero—. Sin ellas no hay cifra de control posible y la limpieza no se puede auditar.
- Esta semana entra también la propuesta formal del proyecto final con el diagnóstico de madurez
  analítica de la empresa elegida. Una empresa que no puede decirles cuántas filas tiene su tabla de
  ventas está, por definición, en el primer nivel de madurez: escríbanlo así.